In [0]:
import joblib
import pandas as pd
from pyspark.sql import functions as F

# Load scalers
scalers = joblib.load('/Workspace/Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/feature engineering/training_temp_scalers.joblib')

def calculate_effort_labels(df_spark, table_name):
    """
    Calculate effort labels (PUSH/MANAGE/CONSERVE) for a Spark DataFrame.
    Returns a Spark DataFrame with one-hot encoded effort columns.
    
    Each dataset (train/validation) computes its own distributions independently
    since these are retrospective labels, not prediction targets.
    """
    # Convert to pandas for processing
    df = df_spark.toPandas()
    
    print(f"\nProcessing {table_name}: {len(df)} rows")
    
    # Check if required columns exist
    required_cols = ['Driver_idx', 'Compound', 'Circuit_Name', 'ThrottleCommitment', 'LiftAndCoastDist', 'AvgBrakeIntensity']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"Warning: Missing columns {missing_cols} in {table_name}")
        return None

    # Denormalize the features using the scalers
    # The gaussian scaler was fitted on ALL gaussian features together
    gaussian_cols = [
        'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 
        'TimeSinceLastWeatherMeasurement',
        'Circuit_CircuitLength', 'Circuit_Number_of_Laps',
        'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs', 'Circuit_AverageAngle',
        'ThrottleCommitment', 'LiftAndCoastDist', 'AvgBrakeIntensity'
    ]
    
    telemetry_cols = ['ThrottleCommitment', 'LiftAndCoastDist', 'AvgBrakeIntensity']
    
    if 'gaussian' in scalers:
        # Get the gaussian scaler
        ss_gauss = scalers['gaussian']
        
        # Find which gaussian columns exist in the data
        existing_gaussian_cols = [c for c in gaussian_cols if c in df.columns]
        
        if existing_gaussian_cols:
            # Denormalize ALL gaussian features together (they were normalized as a group)
            denormalized = ss_gauss.inverse_transform(df[existing_gaussian_cols])
            
            # Extract just the telemetry columns we need
            for i, col in enumerate(existing_gaussian_cols):
                if col in telemetry_cols:
                    df[f'{col}_denorm'] = denormalized[:, i]
            
            print(f"  ✓ Denormalized {len(existing_gaussian_cols)} gaussian features, extracted telemetry: {[c for c in telemetry_cols if c in existing_gaussian_cols]}")
        else:
            print(f"  Warning: No gaussian columns found in data")
            for col in telemetry_cols:
                df[f'{col}_denorm'] = 0  # Fallback
    else:
        print(f"  Warning: 'gaussian' scaler not found, using original values")
        for col in telemetry_cols:
            if col in df.columns:
                df[f'{col}_denorm'] = df[col]
            else:
                df[f'{col}_denorm'] = 0
    
    # BUG FIX 1: Filter SC/VSC laps before computing p95 (they inflate LAC p95)
    clean_mask = df['status_1'] == 1
    
    # Circuit-relative normalization: divide by circuit 95th percentile
    # Compute p95 on CLEAN laps only (no SC/VSC) - independently for each dataset
    for col in ['ThrottleCommitment', 'LiftAndCoastDist', 'AvgBrakeIntensity']:
        denorm_col = f'{col}_denorm'
        # Compute p95 per circuit on clean laps only
        p95_map = df[clean_mask].groupby('Circuit_Name')[denorm_col].quantile(0.95).to_dict()
        # Apply to ALL laps (including SC laps for sequence context)
        df[f'{col}_circuit_norm'] = df[denorm_col] / df['Circuit_Name'].map(p95_map).fillna(1.0)
    
    print(f"  ✓ Computed circuit p95 on {clean_mask.sum()} clean laps (out of {len(df)} total)")
    
    # BUG FIX 2: Make groupby levels consistent
    # Z-score components within driver × compound × circuit (tighter within-circuit labels)
    components = {
        'LiftAndCoastDist_circuit_norm': -1.0,    # lower LAC = more push
        'ThrottleCommitment_circuit_norm': 1.0,   # higher WOT = more push
        'AvgBrakeIntensity_circuit_norm': 1.0     # sharper braking = more push
    }
    
    for col in components.keys():
        # Z-score within Driver × Compound × Circuit (consistent with threshold groupby)
        df[f'{col}_z'] = df.groupby(['Driver_idx', 'Compound'])[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-6)
        )
    
    # Calculate effort score
    df['effort'] = (
        -1.0 * df['LiftAndCoastDist_circuit_norm_z'] +
        1.0 * df['ThrottleCommitment_circuit_norm_z'] +
        1.0 * df['AvgBrakeIntensity_circuit_norm_z']
    )
    
    # Set thresholds relative to each driver's own baseline per compound per circuit
    df['effort_mean'] = df.groupby(['Driver_idx', 'Compound'])['effort'].transform('mean')
    df['effort_std'] = df.groupby(['Driver_idx', 'Compound'])['effort'].transform('std')
    
    # Apply thresholds: mean ± 0.5 * std
    df['label'] = 'MANAGE'
    df.loc[df['effort'] > df['effort_mean'] + 0.5 * df['effort_std'], 'label'] = 'PUSH'
    df.loc[df['effort'] < df['effort_mean'] - 0.5 * df['effort_std'], 'label'] = 'CONSERVE'
    
    # One-hot encode the effort labels
    df['effort_PUSH'] = (df['label'] == 'PUSH').astype(int)
    df['effort_MANAGE'] = (df['label'] == 'MANAGE').astype(int)
    df['effort_CONSERVE'] = (df['label'] == 'CONSERVE').astype(int)
    
    print(f"  Label distribution: PUSH={df['effort_PUSH'].sum()}, MANAGE={df['effort_MANAGE'].sum()}, CONSERVE={df['effort_CONSERVE'].sum()}")
    
    # Drop the original columns and intermediate calculation columns
    cols_to_drop = ['ThrottleCommitment', 'LiftAndCoastDist', 'AvgBrakeIntensity',
                    'ThrottleCommitment_denorm', 'LiftAndCoastDist_denorm', 'AvgBrakeIntensity_denorm',
                    'ThrottleCommitment_circuit_norm', 'LiftAndCoastDist_circuit_norm', 'AvgBrakeIntensity_circuit_norm',
                    'LiftAndCoastDist_circuit_norm_z', 'ThrottleCommitment_circuit_norm_z', 'AvgBrakeIntensity_circuit_norm_z',
                    'effort', 'effort_mean', 'effort_std', 'label']
    
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    
    # Convert back to Spark DataFrame
    return spark.createDataFrame(df)

# Process raw_training (compute p95 independently)
print("=" * 60)
print("Processing raw_training table")
print("=" * 60)
raw_training = spark.table('f1_racing_laptime_pred.raw_training')
silver_training = calculate_effort_labels(raw_training, 'raw_training')

if silver_training is not None:
    # Write to silver_training table (overwrite schema since we changed columns)
    silver_training.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('f1_racing_laptime_pred.silver_training')
    print(f"✓ Written to f1_racing_laptime_pred.silver_training")
    print(f"  Total columns: {len(silver_training.columns)}")
    print(f"  Total rows: {silver_training.count()}")

# Process raw_validating (compute p95 independently)
print("\n" + "=" * 60)
print("Processing raw_validating table")
print("=" * 60)
raw_validating = spark.table('f1_racing_laptime_pred.raw_validating')
silver_validating = calculate_effort_labels(raw_validating, 'raw_validating')

if silver_validating is not None:
    # Write to silver_validating table (overwrite schema since we changed columns)
    silver_validating.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('f1_racing_laptime_pred.silver_validating')
    print(f"✓ Written to f1_racing_laptime_pred.silver_validating")
    print(f"  Total columns: {len(silver_validating.columns)}")
    print(f"  Total rows: {silver_validating.count()}")

print("\n" + "=" * 60)
print("✓ Processing complete!")
print("=" * 60)

In [0]:
# Add race_completedness feature to silver tables
import joblib
import pandas as pd
from pyspark.sql import functions as F

print("=" * 80)
print("Adding race_completedness feature to silver tables")
print("=" * 80)

# Load scalers to denormalize features
scalers = joblib.load('/Workspace/Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/feature engineering/training_temp_scalers.joblib')

def add_race_completedness(df_spark, table_name):
    """
    Add race_completedness feature by denormalizing LapNumber and Circuit_Number_of_Laps,
    then computing the ratio.
    """
    # Convert to pandas for denormalization
    df = df_spark.toPandas()
    
    print(f"\nProcessing {table_name}: {len(df)} rows")
    
    # The gaussian scaler was fitted on ALL gaussian features together
    gaussian_cols = [
        'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 
        'TimeSinceLastWeatherMeasurement',
        'Circuit_CircuitLength', 'Circuit_Number_of_Laps',
        'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs', 'Circuit_AverageAngle',
        'LapNumber'  # LapNumber is also gaussian-normalized
    ]
    
    # Denormalize LapNumber using minmax scaler
    if 'minmax' in scalers:
        mm_scaler = scalers['minmax']
        feature_names = mm_scaler.feature_names_in_
        
        if 'LapNumber' in feature_names and 'LapNumber' in df.columns:
            idx = list(feature_names).index('LapNumber')
            data_min = mm_scaler.data_min_[idx]
            data_max = mm_scaler.data_max_[idx]
            df['LapNumber_denorm'] = df['LapNumber'] * (data_max - data_min) + data_min
            print(f"  ✓ Denormalized LapNumber (min={data_min:.2f}, max={data_max:.2f})")
        elif 'LapNumber' in df.columns:
            print(f"  Warning: LapNumber not in minmax scaler, using original values")
            df['LapNumber_denorm'] = df['LapNumber']
        else:
            print(f"  Warning: LapNumber not in data")
            df['LapNumber_denorm'] = 0
    elif 'LapNumber' in df.columns:
        print(f"  Warning: minmax scaler not found, using original values")
        df['LapNumber_denorm'] = df['LapNumber']
    else:
        print(f"  Warning: LapNumber not in data")
        df['LapNumber_denorm'] = 0
    
    # Denormalize Circuit_Number_of_Laps using gaussian scaler
    if 'gaussian' in scalers:
        ss_gauss = scalers['gaussian']
        feature_names = ss_gauss.feature_names_in_
        
        if 'Circuit_Number_of_Laps' in feature_names and 'Circuit_Number_of_Laps' in df.columns:
            idx = list(feature_names).index('Circuit_Number_of_Laps')
            mean_val = ss_gauss.mean_[idx]
            scale_val = ss_gauss.scale_[idx]
            df['Circuit_Number_of_Laps_denorm'] = df['Circuit_Number_of_Laps'] * scale_val + mean_val
            print(f"  ✓ Denormalized Circuit_Number_of_Laps (mean={mean_val:.2f}, scale={scale_val:.2f})")
        elif 'Circuit_Number_of_Laps' in df.columns:
            print(f"  Warning: Circuit_Number_of_Laps not in scaler, using original values")
            df['Circuit_Number_of_Laps_denorm'] = df['Circuit_Number_of_Laps']
        else:
            print(f"  Warning: Circuit_Number_of_Laps not in data")
            df['Circuit_Number_of_Laps_denorm'] = 0
    else:
        print(f"  Warning: 'gaussian' scaler not found")
        if 'Circuit_Number_of_Laps' in df.columns:
            df['Circuit_Number_of_Laps_denorm'] = df['Circuit_Number_of_Laps']
        else:
            df['Circuit_Number_of_Laps_denorm'] = 0
    
    # Compute race_completedness from denormalized values
    df['race_completedness'] = df['LapNumber_denorm'] / df['Circuit_Number_of_Laps_denorm']
    
    print(f"  race_completedness range: [{df['race_completedness'].min():.4f}, {df['race_completedness'].max():.4f}]")
    
    # Drop intermediate denormalized columns
    df = df.drop(columns=['LapNumber_denorm', 'Circuit_Number_of_Laps_denorm'])
    
    # Convert back to Spark DataFrame
    return spark.createDataFrame(df)

# Process silver_training
print("\n" + "=" * 60)
print("Processing silver_training")
print("=" * 60)
training = spark.table('f1_racing_laptime_pred.silver_training')
training_with_completedness = add_race_completedness(training, 'silver_training')

if training_with_completedness is not None:
    training_with_completedness.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('f1_racing_laptime_pred.silver_training')
    print(f"✓ Updated silver_training")
    print(f"  Total columns: {len(training_with_completedness.columns)}")
    print(f"  Total rows: {training_with_completedness.count()}")

# Process silver_validating
print("\n" + "=" * 60)
print("Processing silver_validating")
print("=" * 60)
validating = spark.table('f1_racing_laptime_pred.silver_validating')
validating_with_completedness = add_race_completedness(validating, 'silver_validating')

if validating_with_completedness is not None:
    validating_with_completedness.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('f1_racing_laptime_pred.silver_validating')
    print(f"✓ Updated silver_validating")
    print(f"  Total columns: {len(validating_with_completedness.columns)}")
    print(f"  Total rows: {validating_with_completedness.count()}")

print("\n" + "=" * 80)
print("✓ race_completedness feature added to both tables!")
print("=" * 80)
print("\nNOTE: race_completedness is computed from DENORMALIZED values.")
print("It represents the actual % of race completed (0% to 100%).")

In [0]:
import joblib
import pandas as pd
from pyspark.sql import functions as F, Window

print("=" * 80)
print("Adding normalized position feature")
print("=" * 80)

# Load scalers to denormalize laptime
scalers = joblib.load('/Workspace/Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/feature engineering/training_temp_scalers.joblib')

def add_position_features(silver_df, table_name):
    """
    Compute normalized position based on denormalized cumulative laptime ranking per circuit and lap.
    Prev_LapTime already exists in the silver tables.
    """
    print(f"\nProcessing {table_name}")
    
    # Convert to pandas for processing
    silver_pandas = silver_df.toPandas()
    
    print(f"  Silver rows: {len(silver_pandas)}")
    
    # Denormalize LapTime_sec
    if 'LapTime_sec' not in silver_pandas.columns:
        print(f"  Error: LapTime_sec not in silver table")
        return None
    
    if 'minmax' in scalers:
        mm_scaler = scalers['minmax']
        feature_names = mm_scaler.feature_names_in_
        
        if 'LapTime_sec' in feature_names:
            idx = list(feature_names).index('LapTime_sec')
            data_min = mm_scaler.data_min_[idx]
            data_max = mm_scaler.data_max_[idx]
            silver_pandas['LapTime_sec_denorm'] = (
                silver_pandas['LapTime_sec'] * (data_max - data_min) + data_min
            )
            print(f"  ✓ Denormalized LapTime_sec (min={data_min:.2f}, max={data_max:.2f})")
        else:
            print(f"  Warning: LapTime_sec not in minmax scaler, using original")
            silver_pandas['LapTime_sec_denorm'] = silver_pandas['LapTime_sec']
    else:
        print(f"  Warning: minmax scaler not found, using original")
        silver_pandas['LapTime_sec_denorm'] = silver_pandas['LapTime_sec']
    
    # Compute cumulative laptime per driver per circuit
    # Sort by Driver_idx, Circuit_Name, LapNumber, then cumsum
    silver_pandas = silver_pandas.sort_values(['Driver_idx', 'Circuit_Name', 'LapNumber'])
    silver_pandas['cumulative_laptime'] = silver_pandas.groupby(
        ['Driver_idx', 'Circuit_Name']
    )['LapTime_sec_denorm'].cumsum()
    
    print(f"  ✓ Computed cumulative laptime per driver")
    print(f"    cumulative_laptime range: [{silver_pandas['cumulative_laptime'].min():.2f}, {silver_pandas['cumulative_laptime'].max():.2f}]")
    
    # Compute position ranking per circuit and lap number
    # Rank drivers by cumulative laptime (lower time = better position = lower rank)
    silver_pandas['position_rank'] = silver_pandas.groupby(
        ['Circuit_Name', 'LapNumber']
    )['cumulative_laptime'].rank(method='min', ascending=True)
    
    # Count active drivers at each circuit and lap
    silver_pandas['active_drivers'] = silver_pandas.groupby(
        ['Circuit_Name', 'LapNumber']
    )['Driver_idx'].transform('count')
    
    # Normalize position: actual rank / number of active drivers
    silver_pandas['position_normalized'] = (
        silver_pandas['position_rank'] / silver_pandas['active_drivers']
    )
    
    print(f"  ✓ Computed position features")
    print(f"    position_rank range: [{silver_pandas['position_rank'].min():.0f}, {silver_pandas['position_rank'].max():.0f}]")
    print(f"    active_drivers range: [{silver_pandas['active_drivers'].min():.0f}, {silver_pandas['active_drivers'].max():.0f}]")
    print(f"    position_normalized range: [{silver_pandas['position_normalized'].min():.4f}, {silver_pandas['position_normalized'].max():.4f}]")
    
    # Drop intermediate columns
    silver_pandas = silver_pandas.drop(columns=['LapTime_sec_denorm', 'cumulative_laptime', 'position_rank', 'active_drivers'])
    
    # Convert back to Spark DataFrame
    return spark.createDataFrame(silver_pandas)

# Process training data
print("\n" + "=" * 60)
print("Processing silver_training")
print("=" * 60)
silver_training = spark.table('f1_racing_laptime_pred.silver_training')
training_updated = add_position_features(silver_training, 'training')

if training_updated is not None:
    training_updated.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('f1_racing_laptime_pred.silver_training')
    print(f"✓ Updated silver_training")
    print(f"  Total columns: {len(training_updated.columns)}")
    print(f"  Total rows: {training_updated.count()}")

# Process validation data
print("\n" + "=" * 60)
print("Processing silver_validating")
print("=" * 60)
silver_validating = spark.table('f1_racing_laptime_pred.silver_validating')
validating_updated = add_position_features(silver_validating, 'validating')

if validating_updated is not None:
    validating_updated.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('f1_racing_laptime_pred.silver_validating')
    print(f"✓ Updated silver_validating")
    print(f"  Total columns: {len(validating_updated.columns)}")
    print(f"  Total rows: {validating_updated.count()}")

print("\n" + "=" * 80)
print("✓ Position normalized feature added to both tables!")
print("=" * 80)
print("\nFeature added:")
print("  - position_normalized: driver position / active drivers at that lap")
print("    (computed from denormalized cumulative laptime ranking)")
print("\nNote: Prev_LapTime already exists in the silver tables")